# Atelier Prompt Engineering — AI Business Assistant 
Ce notebook regroupe l'ensemble des réponses à l'atelier, organisées partie par partie.
## Contexte

L’entreprise souhaite mettre en place **AI Business Assistant**, un assistant IA polyvalent pour exploiter des documents, analyser des données, faire du machine learning et produire des résultats structurés. L’objectif de l’atelier est de construire progressivement et d’améliorer les prompts permettant de réaliser ces différentes tâches.

# Partie 1 — Anatomie d’un prompt

### Problème
Analyser les retours de clients d’une entreprise.

**Décomposition en composantes** : rôle, contexte, tâche, contraintes/format.


**Rôle**: Tu es un analyste spécialisé dans l’analyse des retours clients.

**Contexte** : Une entreprise de e‑commerce reçoit chaque semaine un grand volume d’avis clients provenant de différentes sources — réseaux sociaux, courriels et formulaires de satisfaction. Ces retours, bien que riches en informations, ne font pas encore l’objet d’une analyse systématique.

**Tâche**: Analyse l’ensemble des avis clients fournis ci‑dessous et produis une synthèse structurée mettant en évidence les tendances, les points forts et les axes d’amélioration.

**Contraintes**: 
1.Indique pour chaque avis le sentiment général : positif, négatif ou neutre.

2.Regroupe les avis par sujet : livraison, produit, service client, prix, application.

3.Mets en avant les points positifs les plus cités et les problèmes les plus fréquents.

4.Résume les tendances principales observées dans les retours.

5.Propose deux ou trois actions concrètes pour améliorer l’expérience client.

6.Utilise uniquement les informations présentes dans les avis, sans ajout ni interprétation.

7.Rédige de façon claire, neutre et concise, sans jugement personnel.

**Format de sortie** :
1. Points positifs
2. Problèmes principaux
3. Tendances
4. Recommandations

----
## Partie 2 — Comparer les techniques de prompting

**Objectif** : tester et comparer les résultats obtenus avec différentes approches de prompt (zero‑shot, one‑shot, few‑shot et prompt structuré).

**Tâche** : classer le commentaire suivant selon le sentiment exprimé :

#### <mark style= "background-color: lightblue">**Zero-shot** </mark>
📋**Prompt:**

Classe le commentaire suivant parmi : positif, négatif, neutre.
Commentaire : « Le service est rapide mais l'application plante régulièrement. »

💬**Réponse du LLM**: <mark> Neutre </mark>


**Evaluation**: Sans exemple ni règle de décision, le modèle traite les deux informations (rapidité / plantages) comme équivalentes en poids et choisit la catégorie "neutre" par défaut face à l'ambiguïté, faute d'indication sur comment trancher.



#### <mark style= "background-color: lightblue">**One-shot**</mark>
📋**Prompt:**

Exemple :

Commentaire : « Le livreur a été très aimable et ponctuel. »
Classe : positif

Maintenant, classe le commentaire suivant parmi positif, négatif ou neutre :
« Le service est rapide mais l'application plante régulièrement. »

💬**Réponse du LLM:**<mark>négatif</mark>

 **Evaluation:** L'exemple positif unique manque de nuance : il n'apprend pas au modèle à peser le pour et le contre d'un avis mixte, ce qui le pousse à privilégier la critique technique (« plante ») et à trancher arbitrairement vers le négatif.




#### <mark style= "background-color: lightblue">**Few-shot** </mark>
📋**Prompt** : 

Classe chaque texte en positif, négatif ou neutre.

Texte : « Le livreur a été très aimable et ponctuel. » → positif

Texte : « Ma commande n’est jamais arrivée. » → négatif

Texte : « Le produit correspond à la description, rien de plus. » → neutre

Texte : « Prix correct mais le service client ne répond jamais. » → négatif

Texte : « Le service est rapide mais l’application plante régulièrement. » → ?

💬**Réponse du LLM:** <mark>Neutre</mark>

**Evaluation:** Grâce aux multiples exemples, le modèle comprend mieux la tâche, mais il hésite toujours sur les avis partagés. Comme le prompt ne lui dit pas si un bug technique est plus grave qu'un service rapide, le modèle choisit neutre au hasard plutôt que négatif.

#### <mark style= "background-color: lightblue">**Prompt structuré** </mark>

📋**Prompt**

**Rôle**: Tu es un système d'analyse de sentiment client pour une entreprise de services numériques.

**Contexte**: Les commentaires clients peuvent contenir plusieurs aspects, parfois contradictoires (points positifs et négatifs dans le même commentaire).

**Tâche** : Classer le commentaire suivant dans une seule catégorie parmi : positif, négatif, neutre.

**Règles de décision** :
- Si le commentaire est globalement favorable sans défaut majeur → positif
- Si le commentaire mentionne un défaut qui impacte fortement l'usage (bug bloquant, panne, dysfonctionnement récurrent), même en présence d'un point positif → négatif
- Si les aspects positifs et négatifs sont équivalents en importance, sans dominance claire → neutre

**Format de sortie** : un seul mot parmi "positif", "négatif" ou "neutre", sans explication ni ponctuation.

**Commentaire à classer** : « Le service est rapide mais l'application plante régulièrement. »

--

💬**Réponse du LLM:** <mark>Négatif</mark>


**Évaluation (Prompt structuré)**

* **Points forts :** L'ajout de règles de décision explicites enlève toute ambiguïté sur la gestion des avis mixtes. Le modèle applique directement la règle stipulant qu'un dysfonctionnement récurrent ("plante régulièrement") implique une classification **négatif**.
* **Respect des contraintes :** Le format imposé (un seul mot, sans explication ni ponctuation) est parfaitement suivi, facilitant une exploitation automatique.
* **Bilan :** C'est la méthode la plus fiable et constante pour cadrer le comportement du modèle sur des tâches complexes.

---
## Partie 3 — Prompt Engineering et raisonnement

### 1. Décomposition du prompt
 « Analyse ces avis clients et donne-moi les problèmes les plus importants ainsi que les recommandations » manque de structure. Voici son anatomie :
 
 -  Tâche principal (Instruction) : Analyser des avis clients, identifier des problèmes et suggérer des actions. 
 - Entrée (Données) : Absente (le texte d'avis clients à analyser n'est pas fourni). 
 - Rôle / Persona : Absent (pas -de rôle d'analyste expérience client spécifié).
 - Contraintes et Format : Absents (aucun format précisé comme JSON ou puces, pas de limite de mots, pas de définition de ce qu'est un problème "important").

### 2.1 Prompt amélioré
📋**Prompt**

Rôle : Tu es un analyste chargé d'exploiter des avis clients pour une entreprise de e-commerce.

Tâche : À partir des avis fournis ci-dessous, identifie les 3 problèmes les plus fréquemment mentionnés et propose une recommandation actionnable pour chacun.

Contraintes :
- Base-toi uniquement sur le contenu des avis fournis, sans aucune exception.
- N'invente aucune information, cause, chiffre ou détail absent des avis.
- Ne suppose aucune intention ou émotion du client qui ne soit pas explicitement exprimée.
- Si moins de 3 problèmes distincts sont identifiables, n'en liste que le nombre réellement présent — ne complète pas artificiellement.
- Chaque problème cité doit être accompagné d'une référence à l'avis correspondant (numéro de l'avis).
- Chaque recommandation doit être directement liée au problème identifié, sans généralité vague (ex : interdiction de recommandations du type "améliorer la qualité de service" sans précision).
- N'utilise aucune connaissance externe sur l'entreprise, le secteur ou des cas similaires.

Format de sortie : Liste numérotée. Pour chaque élément : Problème (avec n° d'avis) → Recommandation.

Avis clients :
1. « Livraison en retard de 3 jours, aucune excuse reçue. »
2. « Le produit est de bonne qualité mais l'emballage était endommagé. »
3. « Service client injoignable pendant plus d'une semaine. »
4. « Très satisfait, rien à signaler. »
5. « Le prix a augmenté sans prévenir, c'est frustrant. »

💬**Réponse du LLM:**

1. Problème (Avis 1 & 3) : Retards de livraison sans communication et service client totalement injoignable pendant plus d'une semaine. → Recommandation : Rétablir un canal de communication fonctionnel pour répondre aux clients et envoyer une notification automatique d'excuse lors des retards de livraison.

2. Problème (Avis 2) : Emballages des colis arrivant endommagés lors de la réception. → Recommandation : Renforcer la protection physique des emballages lors du conditionnement avant l'expédition.

3. Problème (Avis 5) : Augmentation des prix appliquée sans notification préalable aux clients. → Recommandation : Informer à l'avance les clients de toute hausse de tarif avant son application.


### 2.2 Vérification par un second prompt
📋**Prompt**

Examine ta réponse précédente à la tâche d'analyse d'avis clients avec la plus grande rigueur, en te basant sur les critères suivants :

1. Informations non justifiées : chaque problème cité est-il explicitement et textuellement présent dans l'avis référencé ? Signale toute reformulation qui dépasse le contenu littéral.
2. Contradictions : y a-t-il des incohérences entre les problèmes identifiés, ou entre un problème et sa recommandation ?
3. Informations absentes : un problème significatif mentionné dans les avis a-t-il été omis ?
4. Hallucinations : y a-t-il des détails, causes, chiffres ou intentions ajoutés qui ne figurent pas dans les avis ?
5. Respect des contraintes : le format (n° d'avis, recommandation spécifique, pas de connaissance externe) a-t-il été strictement respecté ?
6. Neutralité : la réponse a-t-elle évité tout jugement de valeur non fondé sur les avis (ex : qualifier un client de "difficile") ?

Pour chaque critère, indique explicitement : conforme / non conforme, avec justification. Si non conforme, corrige la réponse et indique précisément la modification apportée.

💬**Réponse de LLM:**

**1. Informations non justifiées : Non conforme**

* **Justification :** Le point 1 fusionne les avis 1 et 3 et ajoute « sans communication », alors que l'avis 1 parle uniquement de l'absence d'excuses.
* **Correction :** Séparer le retard de livraison sans excuses (Avis 1) du service client injoignable (Avis 3).

**2. Contradictions : Conforme**

* **Justification :** Chaque recommandation répond directement au problème associé sans incohérence.

**3. Informations absentes : Conforme**

* **Justification :** L'avis 4 (« Très satisfait ») ne contient pas de problème. Tous les avis négatifs (1, 2, 3 et 5) ont été traités.

**4. Hallucinations : Non conforme**

* **Justification :** L'expression « lors du conditionnement avant l'expédition » (point 2) ajoute un détail sur le processus logistique non mentionné dans l'avis.
* **Correction :** Supprimer cette précision et limiter la recommandation au renforcement des emballages.

**5. Respect des contraintes : Conforme**

* **Justification :** Les numéros d'avis, les recommandations ciblées et le format en liste numérotée ont été respectés.

**6. Neutralité : Conforme**

* **Justification :** Aucun jugement sur les clients ni interprétation d'émotions non formulées.


### **Réponse corrigée**

1. Problème (Avis 1) : Livraison en retard de 3 jours sans excuses reçues. → Recommandation : Envoyer un message d'excuse au client en cas de retard de livraison.
2. Problème (Avis 2) : Emballage du produit endommagé à l'arrivée. → Recommandation : Renforcer la protection des emballages utilisés.
3. Problème (Avis 3) : Service client injoignable pendant plus d'une semaine. → Recommandation : Rendre le service client accessible pour répondre aux sollicitations.



---
## Partie 4 — Sorties structurées
###  <mark style= "background-color: grey">**Prompt JSON** </mark>

 <mark style= "background-color: lightblue">**📋Prompt** </mark>


Analyse le commentaire du client et retourne uniquement un objet JSON valide.

Le JSON doit contenir exactement ces 5 champs :
- sentiment : chaîne de caractères, valeur parmi "positif", "negatif", "neutre"
- categorie : chaîne de caractères, par exemple "livraison", "produit", "service" ou "autre"
- urgence : chaîne de caractères, valeur parmi "faible", "moyenne", "élevée"
- probleme : chaîne de caractères décrivant brièvement le problème
- confiance : nombre décimal compris entre 0 et 1

Commentaire :
"Le client est mécontent car sa commande est arrivée avec trois jours de retard."

Ne retourne aucun texte en dehors du JSON.

 <mark style= "background-color: lightblue">💬**Réponse du LLM:** </mark>


{
  "sentiment": "negatif",

  "categorie": "livraison",

  "urgence": "moyenne",

  "probleme": "Retard de livraison de trois jours",
  
  "confiance": 0.95
}

#### <mark style= "background-color: grey">**Régles de validation** </mark>

1. La réponse doit être un JSON valide.
2. Elle doit contenir exactement les 5 propriétés demandées.
3. Aucune propriété supplémentaire n'est autorisée.
4. sentiment doit respecter les valeurs autorisées.
5. urgence doit respecter les valeurs autorisées.
6. confiance doit être comprise entre 0 et 1.


---
## Partie 5 — Prompts pour les applications métier

### 1. Résumer un document
 <mark style= "background-color: lightblue">**📋Prompt** </mark>

Rôle : Tu es un assistant chargé de résumer des documents professionnels.

Tâche : Résume le document fourni ci-dessous en respectant les contraintes suivantes :

Longueur maximale : 250 mots ;
Conserver uniquement les informations factuelles présentes dans le document ;
Identifier clairement les objectifs mentionnés ;
Identifier clairement les résultats mentionnés ;
Identifier clairement les recommandations mentionnées ;
N'inventer aucune information absente du document.

Format de réponse :
Objectifs :
Résultats :
Recommandations :
Résumé :

Document :
"""
DOCUMENT À ANALYSER
"""

 <mark style= "background-color: lightblue">💬**Réponse du LLM:** </mark>

**Objectifs :**

* Automatiser le résumé de documents, l'extraction d'informations, la classification des demandes et la traduction de contenus professionnels.


* Améliorer la rapidité du traitement des demandes internes.



**Résultats :**

* Un prototype a été testé sur 120 documents professionnels.


* Le temps moyen de première analyse par document est passé de 18 minutes à 6 minutes.


* Les utilisateurs ayant participé aux tests ont rapporté une meilleure organisation des informations.



**Recommandations :**

* Poursuivre les tests sur des types de documents plus variés.


* Ajouter une étape de validation pour les sorties structurées.


* Mettre en place une vérification humaine pour les informations importantes avant toute utilisation.



**Résumé :**
Dans le cadre du projet AI Business Assistant, l'entreprise cherche à réduire le temps consacré aux tâches répétitives et à faciliter l'accès aux données internes pour ses collaborateurs. Le projet vise à automatiser le traitement des documents (résumé, extraction, classification, traduction) et à accélérer le traitement des demandes internes. Les tests menés sur un prototype avec 120 documents ont permis de réduire le temps moyen d'analyse par document de 18 à 6 minutes, tout en apportant une meilleure organisation des informations aux utilisateurs. Pour la suite, il est préconisé d'étendre les tests à des documents plus variés, d'ajouter une validation des sorties structurées et d'instaurer une vérification humaine obligatoire pour les données critiques.

### 2. Prompt – Traduction français → anglais
<mark style= "background-color: lightblue">**📋Prompt** </mark>

**Rôle** : Tu es un traducteur professionnel spécialisé dans les documents techniques et métier.

**Tâche** : Traduis le document français fourni ci-dessous vers l'anglais, en respectant strictement les contraintes suivantes :

Conserver intégralement le sens du texte source ;
Conserver la structure du document (paragraphes, listes, titres, ordre des sections) ;
Conserver les termes techniques dans leur forme appropriée en anglais, sans les traduire de façon approximative ;
Ne pas résumer le contenu ; traduire l'intégralité du texte ;
Ne rajouter aucune information absente du texte source.

Document à traduire :

"""
DOCUMENT À TRADUIRE
"""

 <mark style= "background-color: lightblue">💬**Réponse du LLM:** </mark>


**User Guide – AI Business Assistant**

**Presentation**

AI Business Assistant helps employees analyze documents, extract information, and prepare professional responses.
**How It Works**
The user provides a document or a request. The system analyzes the available information and produces a response tailored to the requested processing.
**Security**
Sensitive information must be verified before being transmitted to the system. The results generated by the assistant must be checked when they are used to make an important decision.

### 3. Prompt – Classification d'un ticket informatique

<mark style= "background-color: lightblue">**📋Prompt** </mark>

Tu es un assistant chargé de classifier les tickets informatiques.

Analyse le ticket fourni et attribue-lui une seule catégorie parmi :

- réseau
- logiciel
- matériel
- sécurité
- accès
- autre

Choisis la catégorie qui correspond le mieux au problème décrit.

Ajoute une justification courte basée uniquement sur les informations présentes dans le ticket.

Retourne uniquement un JSON valide avec exactement les deux champs suivants :
{
  "categorie": "...",
  "justification": "..."
}

Ticket :
"""
[TICKET A INSERER]
"""

 <mark style= "background-color: lightblue">💬**Réponse du LLM:** </mark>
 
{
 **"categorie"**: "réseau",
 
 **"justification"**: "Le problème concerne l'impossibilité d'accéder au serveur de fichiers via la connexion au réseau Wi-Fi du bureau[cite: 3]."
  
}

### 4. Prompt – Extraction d'informations depuis une facture

<mark style= "background-color: lightblue">**📋Prompt** </mark>

Tu es un assistant spécialisé dans le traitement de factures.
Analyse attentivement la facture ci‑dessous et retourne uniquement un objet JSON valide contenant exactement les six champs suivants :

- numero_facture
- date
- client
- montant_ht
- tva
- montant_ttc

Règles :

- Si une information est absente ou indéterminable, attribue la valeur null.
- Ne devine aucune donnée.
- N’ajoute aucun champ supplémentaire.
- Respecte strictement le format JSON attendu.
- Format attendu :

{
  "numero_facture": null,

  "date": null,

  "client": null,

  "montant_ht": null,

  "tva": null,
  
  "montant_ttc": null
}
Facture :

[FACTURE A INSERER]


 <mark style= "background-color: lightblue">💬**Réponse du LLM:** </mark>

 {
  "numero_facture": "FAC-2026-0187",

  "date": "10/09/2026",

  "client": "Orange Business Client",

  "montant_ht": "500 000 FCFA",

  "tva": "90 000 FCFA",
  
  "montant_ttc": "590 000 FCFA"
   
}

### 4. Prompt – Email client pour livraison en retard

<mark style= "background-color: lightblue">**📋Prompt** </mark>


Rédige uniquement l'email.

Tu es un assistant spécialisé dans la rédaction d’emails professionnels destinés aux clients.
Rédige un email à un client dont la livraison a pris du retard.

L’email doit :

reconnaître explicitement le retard ;

présenter des excuses claires ;

expliquer la situation uniquement à partir des informations fournies ;

ne jamais inventer de cause au retard ;

proposer une solution ou une prochaine étape concrète.

Ton doit être :

professionnel ;

courtois ;

rassurant.

Contraintes :

maximum 150 mots ;

rester clair et naturel ;

ne pas minimiser le retard ;

ne pas inventer de date, de cause ou de compensation non mentionnée.

Informations disponibles :

text
[LES INFORMATIONS SUR LA LIVRAISON ]
Instruction finale :  
Rédige uniquement le contenu de l’email.

<mark style= "background-color: lightblue">💬**Réponse du LLM:** </mark>

Objet : Information concernant le suivi de votre commande n° CMD-2026-0921

Chère Mme Awa Ndiaye,

Nous vous contactons au sujet de votre commande n° CMD-2026-0921, dont la livraison était initialement prévue le 8 septembre 2026.

Nous reconnaissons que votre livraison a pris du retard et nous vous présentons nos plus sincères excuses pour ce désagrément.

Afin de vous tenir informée au mieux, nous vous confirmons que la nouvelle date de livraison est désormais fixée au 12 septembre 2026. Nous mettons tout en œuvre pour que votre colis vous parvienne à cette date.

Nous restons à votre entière disposition si vous souhaitez obtenir des détails complémentaires sur l'acheminement de votre commande.p

Nous vous remercions pour votre compréhension et votre confiance.

Cordialement,

Le service client

## Partie 6 — Prompt Engineering pour le Machine Learning

### 1. Nettoyage du dataset des capteurs

<mark style= "background-color: lightblue">**📋Prompt** </mark>

Tu es un expert en préparation des données pour le Machine Learning.

Analyse le dataset de capteurs fourni concernant la consommation énergétique d’un bâtiment.

Pour chacun des problèmes suivants :

* valeurs manquantes ;
* doublons ;
* valeurs aberrantes ;
* variables catégorielles ;

propose une stratégie complète en précisant :

1. Comment détecter le problème ;
2. Comment le traiter ;
3. Les différentes méthodes possibles ;
4. Les risques ou conséquences d’un mauvais traitement ;
5. La méthode que tu recommandes pour ce dataset et pourquoi.

Ne modifie pas les données et n’invente aucune information absente du dataset

<mark style= "background-color: lightblue">💬**Réponse du LLM:** </mark>
 Analyse du jeu de données `mesures_capteurs.csv` concernant la consommation énergétique du bâtiment :

**1. Valeurs manquantes**

* **Détection :** Utilisation des fonctions `df.isnull().sum()` ou `df.isna()` sur le dataframe, complétées par une inspection visuelle des séries temporelles interrompues. Le dataset contient 6 valeurs manquantes pour `temperature`, 5 pour `humidite`, `pression` et `consommation`, ainsi que 4 pour `etat`.
* **Traitement :** Identification du type de données (numérique continu vs catégoriel) puis choix d'une méthode de suppression ou d'imputation adaptée à la structure temporelle.
* **Méthodes possibles :**
* Suppression des lignes concernées (`dropna`).
* Imputation par statistiques globales (moyenne, médiane ou mode).
* Imputation par modèle d'apprentissage (KNNImputer, MICE).
* Imputation par interpolation temporelle (*linear*, *time*, *forward fill*, *backward fill*).


* **Risques :** La suppression pure et simple casse la continuité des séquences temporelles des capteurs. L'imputation par une moyenne globale masque la dynamique naturelle (cycles jour/nuit) et réduit artificiellement la variance.
* **Recommandation pour ce dataset :**
* Pour les variables numériques (`temperature`, `humidite`, `pression`, `consommation`) : **Interpolation linéaire temporelle** groupée par `id_capteur` après un tri sur `date_heure`. Les données issues de capteurs forment une série chronologique continue ; l'interpolation préserve la cohérence des valeurs environnantes.
* Pour la variable catégorielle (`etat`) : **Imputation par la valeur la plus fréquente (mode)** ou création d'une catégorie dédiée (`"INCONNU"`).



---

**2. Doublons**

* **Détection :** Recherche des lignes rigoureusement identiques via `df.duplicated().sum()`, ou vérification de l'unicité de l'index spatio-temporel via `df.duplicated(subset=['id_capteur', 'date_heure'])`. Le jeu de données compte 5 doublons stricts.
* **Traitement :** Identification des lignes dupliquées puis filtrage pour ne conserver qu'une seule instance valide.
* **Méthodes possibles :**
* Suppression complète des répliques (`drop_duplicates(keep='first')` ou `keep='last'`).
* Agrégation des mesures simultanées par calcul de moyenne.


* **Risques :** Conserver des doublons fausse les métriques d'évaluation du modèle (ex: biais de sur-représentation) et génère des fuites de données (*data leakage*) si le doublon est séparé entre le jeu d'entraînement et le jeu de test.
* **Recommandation pour ce dataset :** **Suppression stricte des doublons** avec `df.drop_duplicates(keep='first')`. Étant donné qu'il s'agit d'enregistrements répétus à l'identique aux mêmes horodatages, leur élimination nettoie la base sans perte d'information utile.

---

**3. Valeurs aberrantes**

* **Détection :** Application de règles métiers (bornes physiques réalistes), calculs statistiques ($Z\text{-score} > 3$ ou écart interquartile $1.5 \times \text{IQR}$) et visualisation par boîte à moustaches (*boxplot*). Le dataset présente :
* Températures anormales : -18.5 °C et 58.7 °C.
* Humidité impossible : 145.0 %.
* Pression anormale : 850.0 hPa.
* Consommation extrême : 875.0 (hors distribution standard).


* **Traitement :** Distinction entre erreur de mesure matérielle et événement réel extrême, puis correction ou élimination.
* **Méthodes possibles :**
* Suppression des enregistrements contenant des aberrations.
* Écrêtage / Winsorisation (remplacement des valeurs au-delà du 1er ou 99e centile).
* Conversion des valeurs aberrantes en `NaN` suivie d'une imputation.


* **Risques :** Une suppression systématique peut supprimer de vrais pics d'anomalies de consommation précieux pour le modèle. L'écrêtage arbitraire modifie la distribution sans garantir la cohérence physique des données de capteur.
* **Recommandation pour ce dataset :** **Remplacement par `NaN` puis interpolation**. Les températures (-18.5 °C / 58.7 °C) et l'humidité (145 %) transgressent les limites physiques observables dans un bâtiment. Les transformer en valeurs manquantes permet de réutiliser la stratégie d'interpolation temporelle par capteur pour reconstruire une valeur réaliste basée sur l'historique proche.

---

**4. Variables catégorielles**

* **Détection :** Inspection des types d'attributs avec `df.select_dtypes(include=['object'])` et analyse de la cardinalité avec `df.nunique()`. Le dataset contient `batiment` (4 catégories), `id_capteur` (12 catégories) et `etat` (3 catégories).
* **Traitement :** Conversion des identifiants et libellés textuels sous un format numérique directement exploitable par les algorithmes de Machine Learning.
* **Méthodes possibles :**
* Encodage One-Hot (*One-Hot Encoding* / variables indicatrices).
* Encodage Ordinal (*Label Encoding*).
* Encodage par fréquence (*Frequency Encoding*) ou par la cible (*Target Encoding*).


* **Risques :** Utiliser un encodage ordinal sur des catégories sans hiérarchie (`batiment` ou `id_capteur`) introduit une relation d'ordre artificielle que l'algorithme interprètera à tort comme une intensité. Utiliser le One-Hot Encoding sur des variables à très forte cardinalité augmente inutilement la dimensionnalité (*curse of dimensionality*).
* **Recommandation pour ce dataset :**
* **One-Hot Encoding** pour `batiment` (4 catégories) et `etat` (3 catégories) : ces variables comportent peu de modalités non ordonnées.
* **Target Encoding** ou **Frequency Encoding** pour `id_capteur` (12 catégories) : cela permet d'intégrer le signal lié au comportement du capteur sans multiplier excessivement le nombre de colonnes dans la matrice de traits.

### 2. Visualisations de la consommation énergétique

<mark style= "background-color: lightblue">**📋Prompt** </mark>

Tu es un analyste de données spécialisé dans la consommation énergétique des bâtiments.

À partir du dataset de capteurs fourni, identifie les visualisations les plus utiles pour comprendre les variations de la consommation énergétique.

Pour chaque visualisation proposée, indique obligatoirement :

* le type de graphique ;
* les variables à utiliser ;
* ce que le graphique permet d’observer ;
* l’objectif de l’analyse ;
* l’interprétation attendue ;
* les éventuels problèmes ou anomalies qu’il pourrait révéler.

Privilégie les graphiques réellement adaptés aux variables présentes dans le dataset. Si une visualisation n’est pas pertinente avec les données disponibles, explique pourquoi au lieu de l’inventer.

Classe les visualisations de la plus importante à la moins importante pour comprendre la consommation énergétique.

<mark style= "background-color: lightblue">💬**Réponse du LLM:** </mark>

![Matrice de corrélation](../images/Code_Generated_Image.png)
![Boite moustache](../images/Code_Generated_Image%20(1).png)
![Nuage de point](../images/Code_Generated_Image%20(2).png)
![Line chart](../images/Code_Generated_Image%20(3).png)

### - Série temporelle de la consommation par bâtiment

* **Type de graphique :** Graphique en lignes (*Line chart* / Série temporelle).
* **Variables à utiliser :** `date_heure` (Axe X), `consommation` (Axe Y), `batiment` ou `id_capteur` (Couleur/Légende).
* **Ce qu'il permet d'observer :** L'évolution chronologique de la consommation énergétique heure par heure pour chaque bâtiment, les cycles journaliers/hebdomadaires, et la tendance globale au cours du mois de janvier 2026.
* **Objectif de l'analyse :** Identifier les motifs de consommation (heures de pointe, heures creuses) et détecter d'éventuelles dérives temporelles ou ruptures de service.
* **Interprétation attendue :** La consommation doit présenter des pics durant les heures d'occupation (journée) et des creux la nuit ou le week-end. Le bâtiment `B004` montre une consommation globale nettement plus élevée que les autres sur l'ensemble de la période.
* **Problèmes ou anomalies révélés :** Révèle des sauts brusques dus à des valeurs aberrantes (ex: un pic ponctuel à 875) ou des trous dans la série liés aux mesures manquantes.

---


### - Nuage de points Consommation vs Température

* **Type de graphique :** Nuage de points (*Scatter plot*).
* **Variables à utiliser :** `temperature` (Axe X), `consommation` (Axe Y), `batiment` (Couleur/Forme des points).
* **Ce qu'il permet d'observer :** La relation directe de dépendance entre la température extérieure/ambiante et la consommation d'énergie du bâtiment.
* **Objectif de l'analyse :** Évaluer l'impact du chauffage ou de la climatisation sur l'appel de charge énergétique.
* **Interprétation attendue :** On observe une corrélation positive modérée (autour de $r = 0.41$) : lorsque la température augmente, la consommation tend à augmenter, ce qui suggère l'usage de systèmes de refroidissement/climatisation dans les bâtiments.
* **Problèmes ou anomalies révélés :** Met en évidence les valeurs de température physiquement absurdes (-18.5 °C ou 58.7 °C) qui s'écartent du nuage principal de points.

---

### - Boîtes à moustaches de la consommation par bâtiment

* **Type de graphique :** Boîte à moustaches (*Boxplot*).
* **Variables à utiliser :** `batiment` (Axe X), `consommation` (Axe Y).
* **Ce qu'il permet d'observer :** La distribution statistique de la consommation (médiane, quartiles $Q1$/$Q3$, étendue) pour chaque bâtiment de manière comparative.
* **Objectif de l'analyse :** Comparer l'intensité énergétique globale et la variabilité entre les différents bâtiments (`B001`, `B002`, `B003`, `B004`).
* **Interprétation attendue :** Le bâtiment `B004` a la médiane de consommation la plus élevée (~280), tandis que `B003` présente la consommation la plus faible (~150) mais avec plusieurs valeurs hors-normes proches de zéro.
* **Problèmes ou anomalies révélés :** Identifie visuellement les points extrêmes (*outliers*) situés au-delà des moustaches (consommations proches de 0 ou extrêmement hautes).

---

### - Carte thermique de corrélation

* **Type de graphique :** Carte thermique (*Heatmap*).
* **Variables à utiliser :** `temperature`, `humidite`, `pression`, `consommation` (Variables numériques).
* **Ce qu'il permet d'observer :** Les coefficients de corrélation linéaire de Pearson entre toutes les variables physiques et la consommation.
* **Objectif de l'analyse :** Sélectionner les variables explicatives clés pour un futur modèle de prédiction/Machine Learning.

* **Interprétation attendue :** La `temperature` est la variable la plus liée à la `consommation` ($r \approx 0.41$). L'`humidite` ($r \approx -0.03$) et la `pression` ($r \approx 0.06$) présentent une corrélation quasi nulle avec la consommation.
* **Problèmes ou anomalies révélés :** Permet de déceler si des erreurs de mesure faussent globalement les corrélations attendues entre variables météorologiques.

---

### - Visualisations non pertinentes ou impossibles avec ce dataset

* **Graphique en camembert (Pie Chart) de répartition géographique / surfacique :**
* **Pourquoi non pertinent :** Nous ne disposons pas de la surface ($m^2$) des bâtiments ni du type d'usage. Faire un camembert de la somme de la consommation sans normaliser par la taille du bâtiment serait trompeur et sans valeur d'analyse énergétique.
* **Courbe de charge selon les jours de la semaine (Semaine vs Week-end) :**
* **Pourquoi limité :** Bien que l'attribut `date_heure` permette de déduire le jour de la semaine, la plage temporelle du dataset couvre moins d'un mois complet (du 5 au 30 janvier 2026). Un graphique agrégé par jour manque de recul pour établir une vraie saisonnalité annuelle.

### 3. Modèles pour prédire la consommation énergétique

<mark style= "background-color: lightblue">**📋Prompt** </mark>

Agis comme un spécialiste du Machine Learning chargé de prédire la consommation énergétique d’un bâtiment.

À partir du dataset de capteurs fourni, propose plusieurs modèles de Machine Learning adaptés à cette tâche.

Pour chaque modèle, présente :

* son nom ;
* le type de problème auquel il correspond ;
* son principe de fonctionnement, expliqué simplement ;
* pourquoi il peut être adapté à ce dataset ;
* ses principaux avantages ;
* ses limites ;
* les métriques permettant d’évaluer ses performances.

Compare ensuite les modèles proposés et recommande celui qui semble le plus pertinent pour ce dataset, en justifiant ton choix.

Ne suppose pas que le dataset contient des variables qui ne sont pas réellement présentes.

<mark style= "background-color: lightblue">💬**Réponse du LLM:** </mark>


### 1. Régression Linéaire (avec Régularisation Ridge/Lasso)

* **Type de problème :** Régression continue supervisée.
* **Principe de fonctionnement :** Elle cherche à établir une relation linéaire directe entre la variable cible ($Y$) et les variables explicatives ($X$). La version Ridge/Lasso ajoute une pénalité sur les coefficients pour éviter le surapprentissage (*overfitting*).
* **Adaptabilité au dataset :** Adaptée en tant que modèle de référence (*baseline*). Elle capture bien la tendance linéaire observée entre la température et la consommation.
* **Avantages :** Simple à entraîner, très rapide, hautement interprétable (les coefficients indiquent directement l'impact de chaque variable).
* **Limites :** Incapable de capturer les relations non linéaires complexes ou les interactions croisées entre variables sans ingénierie de variables poussée.
* **Métriques d'évaluation :** $R^2$, MAE (*Mean Absolute Error*), RMSE (*Root Mean Squared Error*).

---

### 2. Random Forest Regressor (Forêt Aléatoire)

* **Type de problème :** Régression continue supervisée (méthode d'ensemble par Bagging).
* **Principe de fonctionnement :** Le modèle combine les prédictions d'un grand nombre d'arbres de décision indépendants, entraînés sur des sous-ensembles aléatoires de données et de variables, puis calcule la moyenne de leurs prédictions.
* **Adaptabilité au dataset :** Très bien adaptée aux données hybrides (combinaison de mesures physiques continues comme la température et de variables catégorielles comme `batiment` ou `id_capteur`).
* **Avantages :** Gère naturellement les non-linéarités, robuste face aux valeurs isolées et aux bruits, ne nécessite pas de normalisation stricte des variables.
* **Limites :** Moins performante pour extrapoler en dehors de la plage des données d'entraînement (ex: températures jamais vues).
* **Métriques d'évaluation :** RMSE, MAE, MAPE (*Mean Absolute Percentage Error*).

---

### 3. XGBoost / LightGBM Regressor (Gradient Boosting)

* **Type de problème :** Régression continue supervisée (méthode d'ensemble par Boosting).
* **Principe de fonctionnement :** Construit une séquence d'arbres de décision où chaque nouvel arbre cherche spécifiquement à corriger les erreurs (résidus) commises par les arbres précédents.
* **Adaptabilité au dataset :** Particulièrement efficace pour capturer les motifs fins de séries temporelles tabulaires et les variations de charge énergétique selon les plages horaires ou les bâtiments.
* **Avantages :** Précision très élevée, gestion native des données manquantes, excellentes performances sur données tabulaires.
* **Limites :** Nécessite un réglage fin des hyperparamètres (*tuning*) pour éviter le surapprentissage sur un petit volume de données (~600 lignes).
* **Métriques d me d'évaluation :** RMSE, MAE, $R^2$.

---

### 4. Régression à Vecteurs de Support (SVR - Support Vector Regressor)

* **Type de problème :** Régression continue supervisée basée sur des marges.
* **Principe de fonctionnement :** Recherche une fonction qui ne s'écarte pas des valeurs réelles de plus d'une marge $\epsilon$, tout en projetant les données dans un espace de plus forte dimension via un noyau (*kernel* RBF).
* **Adaptabilité au dataset :** Adaptée aux petits jeux de données continus comportant des relations non linéaires douces entre la météo et la consommation.
* **Avantages :** Très efficace sur les datasets de petite taille, robuste aux variables inutiles grâce à la marge d'erreur.
* **Limites :** Nécessite un nettoyage strict, une imputation des données manquantes et un passage à l'échelle (*StandardScaler*) obligatoires.
* **Métriques d'évaluation :** RMSE, MAE.

---

### Comparatif des Modèles

| Modèle | Interprétabilité | Précision attendue | Sensibilité aux outliers | Prérequis de préparation |
| --- | --- | --- | --- | --- |
| **Régression Linéaire** | Excellente | Faible à Moyenne | Élevée | Encodage des catégories + Normalisation |
| **Random Forest** | Moyenne | Élevée | Faible | Encodage des catégories |
| **XGBoost / LightGBM** | Faible à Moyenne | Très Élevée | Moyenne | Encodage des catégories |
| **SVR (Noyau RBF)** | Faible | Moyenne à Élevée | Moyenne | Encodage + Normalisation + Imputation stricte |

---

### Recommandation & Justification

Pour ce jeu de données spécifique, le modèle recommandé est **Random Forest Regressor** (avec une alternative vers **XGBoost** si le dataset venait à s'agrandir).

**Justification du choix :**

1. **Taille du dataset :** Le jeu de données contient 605 enregistrements, ce qui est un volume réduit. Random Forest offre un excellent compromis en évitant le surapprentissage (*overfitting*) grâce au principe du bagging.
2. **Hétérogénéité des données :** Le dataset associe des variables physiques numériques (`temperature`, `humidite`, `pression`) à des identifiants catégoriels (`batiment`, `id_capteur`). Les arbres de décision gèrent naturellement cette dualité après un encodage classique.
3. **Robustesse aux anomalies :** Même après nettoyage, les résidus de variations brusques de consommation sont mieux lissés par un ensemble d'arbres que par une régression linéaire.

### 4. Métriques de classification

<mark style= "background-color: lightblue">**📋Prompt** </mark>

Explique de manière simple et progressive les métriques suivantes utilisées pour évaluer un modèle de classification :

* Accuracy ;
* Precision ;
* Recall ;
* F1-score ;
* ROC-AUC.

Pour chaque métrique, donne :

1. Une définition simple ;
2. Ce que la métrique mesure réellement ;
3. Comment interpréter une valeur élevée ou faible ;
4. Un exemple concret lié à un problème de classification ;
5. Le contexte dans lequel cette métrique est particulièrement utile ;
6. Une limite ou un piège à éviter lors de son utilisation.

Termine par un tableau comparatif permettant de comprendre rapidement la différence entre ces cinq métriques.

Utilise un vocabulaire accessible à un débutant en Machine Learning et évite les explications inutilement mathématiques.

<mark style= "background-color: lightblue">💬**Réponse du LLM:** </mark>

En classification, évaluer un modèle revient à mesurer sa capacité à bien prédire des catégories (par exemple : "Spam" ou "Non Spam", "Malade" ou "Sain"). Pour éviter de se tromper, on s'appuie sur une matrice de confusion qui compte :

* **Vrais Positifs (VP) :** Correctement prédit positif.
* **Vrais Négatifs (VN) :** Correctement prédit négatif.
* **Faux Positifs (FP) :** Prédit positif à tort (fausse alerte).
* **Faux Négatifs (FN) :** Prédit négatif à tort (manqué).

---

### 1. Accuracy (Exactitude)

* **Définition simple :** Le pourcentage de prédictions correctes sur l'ensemble total des cas.
* **Ce qu'elle mesure réellement :** La performance globale du modèle toutes classes confondues.
* **Interprétation :** Une valeur proche de 1 (100 %) signifie que le modèle ne se trompe presque jamais. Une valeur basse signifie qu'il fait beaucoup d'erreurs globales.
* **Exemple concret :** Prédire si une image contient un chat ou un chien. Sur 100 images, le modèle en classe correctement 90. L'accuracy est de 90 %.
* **Contexte utile :** À utiliser uniquement lorsque les classes sont équilibrées (autant de chats que de chiens dans les données).
* **Piège à éviter :** Le **piège des classes déséquilibrées**. Si 99 % des e-mails sont normaux et 1 % sont des spams, un modèle qui prédit toujours "normal" aura 99 % d'accuracy tout en étant totalement inutile pour détecter les spams.

---

### 2. Precision (Précision)

* **Définition simple :** Quand le modèle prédit que c'est positif, à quel point peut-on lui faire confiance ?
* **Ce qu'elle mesure réellement :** La fiabilité des prédictions positives (réduction des fausses alertes).
* **Interprétation :** Une précision élevée signifie que chaque fois que le modèle donne une alerte, elle est presque toujours vraie. Une précision faible indique une abondance de faux positifs.
* **Exemple concret :** Filtre anti-spam. Si le modèle classe un e-mail en "Spam", on veut être sûr qu'il s'agit d'un vrai spam pour ne pas envoyer un e-mail important à la corbeille.
* **Contexte utile :** Utile lorsque le coût d'un **Faux Positif** est très élevé.
* **Piège à éviter :** Le modèle peut devenir trop timide : s'il ne prédit qu'un seul positif dans toute la journée mais qu'il a raison, la précision sera de 100 %, même s'il a manqué des centaines d'autres cas.

---

### 3. Recall (Rappel ou Sensibilité)

* **Définition simple :** Sur l'ensemble des vrais cas positifs qui existent, combien le modèle a-t-il réussi à en capturer ?
* **Ce qu'il mesure réellement :** La capacité du modèle à déceler tous les éléments d'intérêt (réduction des oublis).
* **Interprétation :** Un rappel élevé signifie que le modèle laisse passer très peu de cas réels. Un rappel faible indique qu'il a manqué beaucoup d'éléments importants.
* **Exemple concret :** Dépistage d'une maladie grave. Il est vital de détecter tous les patients malades, quitte à faire repasser des tests de contrôle à des personnes saines.
* **Contexte utile :** Utile lorsque le coût d'un **Faux Négatif** (un oubli) est critique ou dangereux.
* **Piège à éviter :** Un modèle qui prédit "positif" pour absolument tout le monde aura un rappel parfait de 100 %, mais générera une quantité ingérable de fausses alertes.

---

### 4. F1-Score

* **Définition simple :** La moyenne équilibrée entre la Precision et le Recall.
* **Ce qu'il mesure réellement :** Le compromis global entre la fiabilité des alertes (Precision) et la couverture des cas (Recall).
* **Interprétation :** Un F1-score élevé (proche de 1) indique que le modèle atteint un bon équilibre sans sacrifier la précision au profit du rappel, et inversement.
* **Exemple concret :** Détection de fraudes bancaires, où l'on souhaite capturer un maximum de transactions suspectes (Recall) sans bloquer continuellement la carte bancaire de clients légitimes (Precision).
* **Contexte utile :** Essentiel pour comparer plusieurs modèles sur des jeux de données déséquilibrés.
* **Piège à éviter :** Il donne un poids égal à la précision et au rappel. Si votre problème métier accorde beaucoup plus d'importance à l'un qu'à l'autre, le F1-Score classique n'est pas l'indicateur idéal.

---

### 5. ROC-AUC (Aire sous la courbe ROC)

* **Définition simple :** La capacité globale du modèle à distinguer la classe positive de la classe négative, quel que soit le seuil de décision choisi.
* **Ce qu'elle mesure réellement :** Le pouvoir de séparation du modèle.
* **Interprétation :** Une valeur de 1.0 indique une séparation parfaite des deux classes. Une valeur de 0.5 signifie que le modèle fait des prédictions complètement aléatoires (comme pile ou face).
* **Exemple concret :** Évaluation de la solvabilité d'un client bancaire. Le modèle donne un score entre 0 et 1 ; la ROC-AUC mesure la capacité du score à placer les mauvais payeurs au-dessus des bons payeurs.
* **Contexte utile :** Idéal pour évaluer et comparer des algorithmes de manière générale avant d'ajuster le seuil de déclenchement d'une alerte.
* **Piège à éviter :** Peut paraître suroptimiste sur des jeux de données extrêmement déséquilibrés.

---

### Tableau comparatif

| Métrique | Question clé à laquelle elle répond | Priorité absolue | Risque majeur |
| --- | --- | --- | --- |
| **Accuracy** | *"Le modèle s'est-il trompé souvent au global ?"* | Équilibre global des données | Trompeuse si les classes sont déséquilibrées. |
| **Precision** | *"Quand le modèle dit 'Oui', est-ce vraiment 'Oui' ?"* | Minimiser les fausses alertes (FP) | Peut manquer beaucoup de vrais cas positifs. |
| **Recall** | *"Le modèle a-t-il raté des cas importants ?"* | Minimiser les oublis dangereux (FN) | Peut générer un très grand nombre de fausses alertes. |
| **F1-Score** | *"Quel est le bon compromis entre Précision et Rappel ?"* | Équilibre sur données déséquilibrées | Masque quelle métrique est favorisée. |
| **ROC-AUC** | *"À quel point le modèle sépare-t-il bien les 2 classes ?"* | Évaluation globale du classement | Moins représentative sur des déséquilibres extrêmes. |

### 5. Métriques de régression
<mark style= "background-color: lightblue">**📋Prompt** </mark>
Tu es un formateur en Machine Learning et tu dois expliquer à un débutant les métriques utilisées pour évaluer un modèle de régression.

Explique les trois métriques suivantes :

* MAE ;
* MSE ;
* RMSE.

Pour chacune, précise :

* sa définition ;
* ce qu’elle mesure ;
* comment interpréter une valeur faible ou élevée ;
* un exemple concret appliqué à la prédiction de la consommation énergétique d’un bâtiment ;
* dans quel contexte elle est particulièrement utile ;
* ses avantages et ses limites.

Explique également pourquoi le MSE et le RMSE donnent davantage d’importance aux grandes erreurs que le MAE.

Termine par un tableau comparatif MAE / MSE / RMSE permettant de choisir facilement la métrique adaptée selon le besoin.


<mark style= "background-color: lightblue">💬**Réponse du LLM:** </mark>

En régression, le modèle ne prédit pas des catégories, mais des **valeurs numériques continues** (ex: la consommation électrique en kWh). L'évaluation consiste donc à mesurer la distance entre la valeur réelle ($y$) et la valeur prédite par le modèle ($\hat{y}$). Cette différence se nomme l'**erreur de prédiction** ou le **résidu** : $\text{Erreur} = y - \hat{y}$.

---

### 1. MAE – Mean Absolute Error (Erreur Absolue Moyenne)

* **Définition :** La moyenne de la valeur absolue de toutes les erreurs de prédiction.
* **Ce qu'elle mesure :** L'écart moyen linéaire entre les prédictions du modèle et les valeurs réelles, exprimé directement dans la même unité que la variable cible.
* **Interprétation :**
* **Faible :** Le modèle prédit en moyenne des valeurs très proches de la réalité.
* **Élevée :** Le modèle commet des écarts moyens importants sur ses prédictions.


* **Exemple concret :** Si la consommation réelle est de **200 kWh** et que le modèle prédit **210 kWh**, l'erreur absolue est de **10 kWh**. Si sur un mois la MAE est de **15 kWh**, cela signifie qu'en moyenne, les prédictions s'écartent de 15 kWh de la réalité.
* **Contexte utile :** Lorsque toutes les erreurs doivent être traitées avec le même poids (une erreur de 20 kWh compte exactement comme deux erreurs de 10 kWh) et que l'on souhaite une métrique simple à expliquer à un public non technique.
* **Avantages et Limites :**
* **Avantage :** Très intuitive à interpréter et robuste face aux valeurs aberrantes (*outliers*).
* **Limite :** Ne fait pas de différence entre plusieurs petites erreurs et une seule grande erreur critique.



---

### 2. MSE – Mean Squared Error (Erreur Quadratique Moyenne)

* **Définition :** La moyenne du carré de toutes les erreurs de prédiction.
* **Ce qu'elle mesure :** La variance des erreurs de prédiction. Elle pénalise fortement les grands écarts.
* **Interprétation :**
* **Faible :** Le modèle fait très peu d'erreurs et ne commet aucune grande erreur.
* **Élevée :** Le modèle commet des erreurs importantes sur certaines prédictions.


* **Exemple concret :** Si l'écart de prédiction est de **10 kWh**, son erreur carrée vaut $10^2 = 100 \text{ kWh}^2$. Si l'écart est de **30 kWh**, l'erreur carrée monte en flèche à $30^2 = 900 \text{ kWh}^2$.
* **Contexte utile :** Très utilisée lors de la phase d'entraînement (*loss function*) de nombreux algorithmes de Machine Learning (comme la régression linéaire) car sa fonction est lisse et dérivable mathématiquement.
* **Avantages et Limites :**
* **Avantage :** Détecte immédiatement si un modèle commet des erreurs extrêmes d'un point de vue mathématique.
* **Limite :** L'unité du résultat est élevée au carré ($\text{kWh}^2$), ce qui rend la valeur difficilement interprétable sur le plan métier.



---

### 3. RMSE – Root Mean Squared Error (Racine de l'Erreur Quadratique Moyenne)

* **Définition :** La racine carrée de la MSE ($\text{RMSE} = \sqrt{\text{MSE}}$).
* **Ce qu'elle mesure :** L'écart-type des résidus. Elle évalue l'ampleur des erreurs tout en appliquant une pénalité accrue aux gros écarts, mais réexprime le résultat dans l'unité d'origine.
* **Interprétation :**
* **Faible :** Le modèle est précis et commet peu de variations aberrantes.
* **Élevée :** Le modèle génère des pics d'erreurs sporadiques importants.


* **Exemple concret :** Pour une consommation énergétique, une RMSE de **25 kWh** indique le niveau d'erreur type du modèle en tenant compte d'une pénalité plus lourde pour les sous-estimations ou surestimations graves de la charge électrique.
* **Contexte utile :** Lorsque l'on veut conserver une pénalisation forte sur les grands écarts (comme avec le MSE), mais tout en gardant une métrique exprimée en kWh compréhensible par des ingénieurs ou décideurs.
* **Avantages et Limites :**
* **Avantage :** Donne une bonne idée de la précision globale tout en sanctionnant les erreurs d'inattention ou d'anomalies du modèle.
* **Limite :** Reste sensible aux valeurs aberrantes dans les données de test.



---

### Pourquoi le MSE et le RMSE pénalisent-ils davantage les grandes erreurs que le MAE ?

La différence provient de l'**élévation au carré** des erreurs avant d'en calculer la moyenne :

* Un écart de **2 kWh** au carré donne une pénalité de **4**.
* Un écart de **10 kWh** au carré donne une pénalité de **100** (soit 5 fois plus grand en écart, mais **25 fois plus lourd** en pénalité).

**Démonstration numérique :**
Prenons deux scénarios sur 5 mesures de consommation ayant la même somme d'erreurs absolues (12 kWh) :

1. **Erreurs régulières et petites :** $[2, 3, 2, 3, 2]$
* $\text{MAE} = 2.40 \text{ kWh}$
* $\text{MSE} = 6.00 \text{ kWh}^2$
* $\text{RMSE} = 2.45 \text{ kWh}$


2. **Une seule grosse erreur isolée :** $[0, 0, 0, 0, 12]$
* $\text{MAE} = 2.40 \text{ kWh}$ *(inchangée)*
* $\text{MSE} = 28.80 \text{ kWh}^2$ *(multipliée par 4.8)*
* $\text{RMSE} = 5.37 \text{ kWh}$ *(plus du double)*



La MAE traite les deux situations de la même façon, alors que le MSE et la RMSE mettent immédiatement en évidence la présence de la grande erreur de 12 kWh.

---

### Tableau comparatif MAE / MSE / RMSE

| Métrique | Unité de mesure | Sensibilité aux grandes erreurs | Interprétabilité Métier | Usage recommandé |
| --- | --- | --- | --- | --- |
| **MAE** | Même unité que la cible (ex: $\text{kWh}$) | **Faible** (Poids linéaire) | **Excellente** (Moyenne directe des erreurs) | Présentation des résultats à un public non technique, ou si quelques grands écarts sont tolérés. |
| **MSE** | Unité au carré (ex: $\text{kWh}^2$) | **Très Élevée** (Poids quadratique) | **Difficile** (Échelle modifiée) | Optimisation mathématique des modèles pendant la phase d'entraînement. |
| **RMSE** | Même unité que la cible (ex: $\text{kWh}$) | **Élevée** (Poids quadratique ajusté) | **Bonne** (Représente l'écart-type de l'erreur) | Évaluation finale lorsque les erreurs importantes sont coûteuses ou risquées (ex: pics de réseau). |